### https://wandb.ai/tempakash48-iiit-hyderabad/cifar10-colorization?nw=nwusertempakash48 - Link to WANDB Runs

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision
import numpy as np
import matplotlib.pyplot as plt
import wandb
import os
from tqdm import tqdm
import copy

# CONFIGURATION


torch.manual_seed(42)
np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

WANDB_API_KEY = "4b42a6bb8776c07603c0f471b0e7e9ec4f936d66"
WANDB_PROJECT = "cifar10-colorization"

# DATASET

class CIFAR10ColorDataset(Dataset):
    def __init__(self, cifar_dataset, color_centroids):
        self.cifar_dataset = cifar_dataset
        # Ensuring Centroids are in [0, 1] range
        self.color_centroids = color_centroids / 255.0 if color_centroids.max() > 1.0 else color_centroids
        
    def __len__(self):
        return len(self.cifar_dataset)
    
    def rgb_to_grayscale(self, rgb_image):
        """Convert RGB PIL image to grayscale tensor [1, H, W]"""
        gray_image = rgb_image.convert('L')
        gray_array = np.array(gray_image, dtype=np.float32) / 255.0
        return torch.from_numpy(gray_array).unsqueeze(0)
    
    def assign_color_labels(self, rgb_image):
        """Assign each pixel to nearest color centroid"""
        rgb_array = np.array(rgb_image, dtype=np.float32) / 255.0
        H, W, C = rgb_array.shape
        pixels = rgb_array.reshape(-1, 3)
        
        # Distances to all centroids
        distances = np.linalg.norm(
            pixels[:, np.newaxis, :] - self.color_centroids[np.newaxis, :, :], 
            axis=2
        )
        labels = np.argmin(distances, axis=1).reshape(H, W)
        return labels
    
    def __getitem__(self, idx):
        rgb_image, _ = self.cifar_dataset[idx]
        
        gray_tensor = self.rgb_to_grayscale(rgb_image)
        color_labels = self.assign_color_labels(rgb_image)
        color_labels_tensor = torch.from_numpy(color_labels).long()
        
        # Original RGB for visualization
        rgb_array = np.array(rgb_image, dtype=np.float32) / 255.0
        rgb_tensor = torch.from_numpy(rgb_array).permute(2, 0, 1)
        
        return gray_tensor, color_labels_tensor, rgb_tensor


def load_data(color_centroids_path='color_centroids.npy', batch_size=64):
    """Load CIFAR-10 and create dataloaders"""
    # Color centroids
    color_centroids = np.load(color_centroids_path)
    print(f"Loaded centroids - shape: {color_centroids.shape}, range: [{color_centroids.min():.2f}, {color_centroids.max():.2f}]")
    
    # CIFAR-10
    cifar_train_full = torchvision.datasets.CIFAR10(root='./data', train=True, download=True)
    cifar_test = torchvision.datasets.CIFAR10(root='./data', train=False, download=True)
    
    # Spliting train into train (90%) and validation (10%)
    train_size = int(0.9 * len(cifar_train_full))
    val_size = len(cifar_train_full) - train_size
    cifar_train, cifar_val = torch.utils.data.random_split(
        cifar_train_full, [train_size, val_size],
        generator=torch.Generator().manual_seed(42)
    )
    
    # Custom datasets
    train_dataset = CIFAR10ColorDataset(cifar_train, color_centroids)
    val_dataset = CIFAR10ColorDataset(cifar_val, color_centroids)
    test_dataset = CIFAR10ColorDataset(cifar_test, color_centroids)
    
    # Dataloaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    
    # Return normalized centroids
    normalized_centroids = color_centroids / 255.0 if color_centroids.max() > 1.0 else color_centroids
    
    return train_loader, val_loader, test_loader, normalized_centroids


# MODEL ARCHITECTURE

class ColorEncoder(nn.Module):
    def __init__(self, NIC=1, NF=16, NC=24, kernel_size=3):
        super(ColorEncoder, self).__init__()
        
        padding = (kernel_size - 1) // 2
        
        # ENCODER
        self.enc1 = nn.Sequential(
            nn.Conv2d(NIC, NF, kernel_size=kernel_size, padding=padding),
            nn.BatchNorm2d(NF),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        
        self.enc2 = nn.Sequential(
            nn.Conv2d(NF, 2*NF, kernel_size=kernel_size, padding=padding),
            nn.BatchNorm2d(2*NF),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        
        self.enc3 = nn.Sequential(
            nn.Conv2d(2*NF, 4*NF, kernel_size=kernel_size, padding=padding),
            nn.BatchNorm2d(4*NF),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        
        # DECODER
        self.dec1 = nn.Sequential(
            nn.ConvTranspose2d(4*NF, 2*NF, kernel_size=2, stride=2),
            nn.BatchNorm2d(2*NF),
            nn.ReLU(inplace=True)
        )
        
        self.dec2 = nn.Sequential(
            nn.ConvTranspose2d(2*NF, NF, kernel_size=2, stride=2),
            nn.BatchNorm2d(NF),
            nn.ReLU(inplace=True)
        )
        
        self.dec3 = nn.Sequential(
            nn.ConvTranspose2d(NF, NC, kernel_size=2, stride=2),
            nn.BatchNorm2d(NC),
            nn.ReLU(inplace=True)
        )
        
        # Final classifier
        self.classifier = nn.Conv2d(NC, NC, kernel_size=1)
        
    def forward(self, x):
        x1 = self.enc1(x)
        x2 = self.enc2(x1)
        x3 = self.enc3(x2)
        
        x4 = self.dec1(x3)
        x5 = self.dec2(x4)
        x6 = self.dec3(x5)
        
        logits = self.classifier(x6)
        return logits


# TRAINING & EVALUATION

def train_epoch(model, train_loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct_pixels = 0
    total_pixels = 0
    
    for gray_imgs, color_labels, _ in tqdm(train_loader, desc="Training", leave=False):
        gray_imgs = gray_imgs.to(device)
        color_labels = color_labels.to(device)
        
        optimizer.zero_grad()
        logits = model(gray_imgs)
        loss = criterion(logits, color_labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * gray_imgs.size(0)
        pred_labels = torch.argmax(logits, dim=1)
        correct_pixels += (pred_labels == color_labels).sum().item()
        total_pixels += color_labels.numel()
    
    epoch_loss = running_loss / len(train_loader.dataset)
    epoch_acc = correct_pixels / total_pixels
    return epoch_loss, epoch_acc


def validate(model, val_loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct_pixels = 0
    total_pixels = 0
    
    with torch.no_grad():
        for gray_imgs, color_labels, _ in tqdm(val_loader, desc="Validation", leave=False):
            gray_imgs = gray_imgs.to(device)
            color_labels = color_labels.to(device)
            
            logits = model(gray_imgs)
            loss = criterion(logits, color_labels)
            
            running_loss += loss.item() * gray_imgs.size(0)
            pred_labels = torch.argmax(logits, dim=1)
            correct_pixels += (pred_labels == color_labels).sum().item()
            total_pixels += color_labels.numel()
    
    epoch_loss = running_loss / len(val_loader.dataset)
    epoch_acc = correct_pixels / total_pixels
    return epoch_loss, epoch_acc


# VISUALIZATIONS


def labels_to_rgb(labels, centroids):
    """Convert predicted labels to RGB images"""
    labels_np = labels.cpu().numpy()
    B, H, W = labels_np.shape
    # Ensuring centroids are in [0, 1] range
    if centroids.max() > 1.0:
        centroids = centroids / 255.0
    rgb_images = centroids[labels_np]
    return rgb_images


def visualize_colorization(model, data_loader, centroids, device, num_samples=10):
    """Generate colorization examples"""
    model.eval()
    
    # Centroids are normalized
    if centroids.max() > 1.0:
        centroids = centroids / 255.0
    
    gray_list, pred_list, gt_list = [], [], []
    
    with torch.no_grad():
        for gray_imgs, color_labels, rgb_imgs in data_loader:
            if len(gray_list) >= num_samples:
                break
            
            gray_imgs = gray_imgs.to(device)
            logits = model(gray_imgs)
            pred_labels = torch.argmax(logits, dim=1)
            pred_rgb = labels_to_rgb(pred_labels, centroids)
            
            batch_size = gray_imgs.size(0)
            for b in range(min(batch_size, num_samples - len(gray_list))):
                gray_list.append(gray_imgs[b, 0].cpu().numpy())
                pred_list.append(pred_rgb[b])
                gt_list.append(rgb_imgs[b].permute(1, 2, 0).cpu().numpy())
    
    fig, axes = plt.subplots(num_samples, 3, figsize=(12, 3.5*num_samples))
    if num_samples == 1:
        axes = axes.reshape(1, -1)
    
    for i in range(num_samples):
        axes[i, 0].imshow(gray_list[i], cmap='gray', vmin=0, vmax=1)
        axes[i, 0].set_title('Input Grayscale')
        axes[i, 0].axis('off')
        
        axes[i, 1].imshow(np.clip(pred_list[i], 0, 1))
        axes[i, 1].set_title('Predicted Color')
        axes[i, 1].axis('off')
        
        axes[i, 2].imshow(np.clip(gt_list[i], 0, 1))
        axes[i, 2].set_title('Ground Truth')
        axes[i, 2].axis('off')
    
    plt.tight_layout()
    return fig


def plot_loss_curves(train_losses, val_losses):
    """Plot training and validation loss curves"""
    fig, ax = plt.subplots(figsize=(10, 6))
    
    epochs = range(1, len(train_losses) + 1)
    ax.plot(epochs, train_losses, 'b-o', label='Training Loss', markersize=4)
    ax.plot(epochs, val_losses, 'r-s', label='Validation Loss', markersize=4)
    
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_title('Training and Validation Loss')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    return fig


# MAIN TRAINING FUNCTION

def train_model(config, train_loader, val_loader, test_loader, centroids, device, num_epochs=25):
    """Main training function"""
    run = wandb.init(project=WANDB_PROJECT, config=config)
    config = wandb.config
    
    # Model
    model = ColorEncoder(
        NIC=config.NIC,
        NF=config.NF,
        NC=config.NC,
        kernel_size=config.kernel_size
    ).to(device)
    
    # Loss and optimizer
    criterion = nn.CrossEntropyLoss()
    
    if config.optimizer == 'Adam':
        optimizer = optim.Adam(model.parameters(), lr=config.learning_rate)
    else:
        optimizer = optim.SGD(model.parameters(), lr=config.learning_rate, momentum=0.9)
    
    # Training loop
    best_val_loss = float('inf')
    best_model_state = None
    best_epoch = 0
    train_losses, val_losses = [], []
    
    for epoch in range(num_epochs):
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc = validate(model, val_loader, criterion, device)
        
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        
        wandb.log({
            'epoch': epoch + 1,
            'train_loss': train_loss,
            'val_loss': val_loss,
            'train_acc': train_acc,
            'val_acc': val_acc
        })
        
        # Best model
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_state = copy.deepcopy(model.state_dict())
            best_epoch = epoch + 1
    
    # Best model
    model.load_state_dict(best_model_state)
    
    # Test evaluation
    test_loss, test_acc = validate(model, test_loader, criterion, device)
    
    wandb.log({
        'best_val_loss': best_val_loss,
        'best_epoch': best_epoch,
        'test_loss': test_loss,
        'test_acc': test_acc
    })
    
    # Visualizations
    loss_fig = plot_loss_curves(train_losses, val_losses)
    wandb.log({"loss_curves": wandb.Image(loss_fig)})
    plt.close(loss_fig)
    
    color_fig = visualize_colorization(model, test_loader, centroids, device, num_samples=10)
    wandb.log({"colorization_examples": wandb.Image(color_fig)})
    plt.close(color_fig)
    
    # checkpoint
    checkpoint_path = f"checkpoint_{run.id}.pth"
    torch.save({
        'model_state_dict': best_model_state,
        'config': dict(config),
        'best_val_loss': best_val_loss,
        'best_epoch': best_epoch,
        'test_loss': test_loss,
        'test_acc': test_acc
    }, checkpoint_path)
    
    # Logging checkpoint as artifact
    artifact = wandb.Artifact(f'model-{run.id}', type='model')
    artifact.add_file(checkpoint_path)
    wandb.log_artifact(artifact)
    
    wandb.finish()
    
    return model, best_val_loss


# SWEEP CONFIGURATION

def run_sweep():
    """Run hyperparameter sweep"""
    sweep_config = {
        'method': 'bayes',
        'metric': {
            'name': 'best_val_loss',
            'goal': 'minimize'
        },
        'parameters': {
            'learning_rate': {'values': [1e-4, 3e-4, 1e-3, 3e-3]},
            'batch_size': {'values': [32, 64, 128]},
            'NF': {'values': [8, 16, 32]},
            'kernel_size': {'values': [3, 5]},
            'optimizer': {'values': ['SGD', 'Adam']},
            'NIC': {'value': 1},
            'NC': {'value': 24},
            'num_epochs': {'value': 25}
        }
    }
    
    sweep_id = wandb.sweep(sweep_config, project=WANDB_PROJECT)
    return sweep_id


def sweep_train():
    """Training function for sweep"""
    config_defaults = {
        'NIC': 1,
        'NC': 24,
        'num_epochs': 25
    }
    
    run = wandb.init(config=config_defaults)
    config = wandb.config
    
    # Loading data with batch size from config
    train_loader, val_loader, test_loader, centroids = load_data(
        color_centroids_path='color_centroids.npy',
        batch_size=config.batch_size
    )
    
    # Train model
    train_model(config, train_loader, val_loader, test_loader, centroids, device, 
                num_epochs=config.num_epochs)



if __name__ == "__main__":
    wandb.login(key=WANDB_API_KEY)
    
    print("Running initial training (25 epochs)...")
    
    config = {
        'NIC': 1,
        'NF': 16,
        'NC': 24,
        'kernel_size': 3,
        'batch_size': 64,
        'learning_rate': 1e-3,
        'optimizer': 'Adam',
        'num_epochs': 25
    }
    
    train_loader, val_loader, test_loader, centroids = load_data(
        color_centroids_path='color_centroids.npy',
        batch_size=config['batch_size']
    )
    
    model, best_val_loss = train_model(
        config, train_loader, val_loader, test_loader, centroids, device, 
        num_epochs=config['num_epochs']
    )
    
    print(f"Initial training complete. Best validation loss: {best_val_loss:.4f}")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


Running initial training (25 epochs)...
Loaded centroids - shape: (24, 3), range: [12.70, 246.06]


best_epoch,▁
best_val_loss,▁
epoch,▁▁▂▂▂▂▃▃▃▄▄▄▅▅▅▅▆▆▆▇▇▇▇██
test_acc,▁
test_loss,▁
train_acc,▁▅▆▆▇▇▇▇▇▇▇▇▇████████████
train_loss,█▄▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁
val_acc,▁▃▄▅▅▅▆▆▄▆▆▇▆▇▆█▇▇▆██▇█▇▇
val_loss,█▆▅▄▄▄▃▃▄▃▃▂▂▂▂▁▁▂▂▁▁▁▁▁▂
best_epoch,23
best_val_loss,1.71934


Initial training complete. Best validation loss: 1.7193


In [20]:
sweep_id = run_sweep()
print(f"Sweep ID: {sweep_id}")
wandb.agent(sweep_id, function=sweep_train, count=20)

Create sweep with ID: 15wz4rr4
Sweep URL: https://wandb.ai/tempakash48-iiit-hyderabad/cifar10-colorization/sweeps/15wz4rr4
Sweep ID: 15wz4rr4


wandb: Agent Starting Run: hwvcaav0 with config:
wandb: 	NC: 24
wandb: 	NF: 16
wandb: 	NIC: 1
wandb: 	batch_size: 32
wandb: 	kernel_size: 5
wandb: 	learning_rate: 0.003
wandb: 	num_epochs: 25
wandb: 	optimizer: SGD


wandb: 
wandb: 🚀 View run  at: https://wandb.ai/Akash%20Temp/cifar10-colorization/runs/a7xc953k
wandb: Find logs at: wandb/run-20251030_141942-a7xc953k/logs


Loaded centroids - shape: (24, 3), range: [12.70, 246.06]


best_epoch,▁
best_val_loss,▁
epoch,▁▁▂▂▂▂▃▃▃▄▄▄▅▅▅▅▆▆▆▇▇▇▇██
test_acc,▁
test_loss,▁
train_acc,▁▄▅▆▆▆▇▇▇▇▇▇▇▇▇██████████
train_loss,█▅▄▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁
val_acc,▁▄▄▆▅▆▆▇▇▇▇▆▇▇▆▇▆▇██▇███▇
val_loss,█▅▄▃▃▂▂▂▂▂▂▂▂▂▃▂▃▂▁▁▁▁▁▁▁
best_epoch,24
best_val_loss,1.57339


...

Loaded centroids - shape: (24, 3), range: [12.70, 246.06]


best_epoch,▁
best_val_loss,▁
epoch,▁▁▂▂▂▂▃▃▃▄▄▄▅▅▅▅▆▆▆▇▇▇▇██
test_acc,▁
test_loss,▁
train_acc,▁▃▄▅▅▆▆▆▆▇▇▇▇▇▇▇▇████████
train_loss,█▅▄▄▃▃▃▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁
val_acc,▁▂▂▄▅▄▅▆▅▇▇▇▅▆▇▇▂▆▇▆█▆▇██
val_loss,█▆▆▄▄▅▃▃▄▂▂▂▃▃▂▂▆▂▂▂▁▂▁▁▁
best_epoch,25
best_val_loss,1.37551


wandb: Agent Starting Run: qiize13l with config:
wandb: 	NC: 24
wandb: 	NF: 32
wandb: 	NIC: 1
wandb: 	batch_size: 32
wandb: 	kernel_size: 5
wandb: 	learning_rate: 0.003
wandb: 	num_epochs: 25
wandb: 	optimizer: Adam


Loaded centroids - shape: (24, 3), range: [12.70, 246.06]


best_epoch,▁
best_val_loss,▁
epoch,▁▁▂▂▂▂▃▃▃▄▄▄▅▅▅▅▆▆▆▇▇▇▇██
test_acc,▁
test_loss,▁
train_acc,▁▃▄▅▅▆▆▆▆▇▇▇▇▇▇▇█████████
train_loss,█▅▄▄▄▃▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁
val_acc,▁▂▂▄▅▅▆▆▆▅▇▆▆▆▆▆▇▇▆▆▇▆██▇
val_loss,█▇▆▄▄▄▃▃▃▃▂▃▃▂▃▂▂▁▂▃▂▄▁▁▂
best_epoch,23
best_val_loss,1.38086
